In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from politics.utils.completion_helpers import validate_completion
from politics.utils.pct_helpers import calculate_pct_coordinates

N_PER_SCORE = 1
TOPICS = ["Harm_Care", "Faireness_Reciprocity", "Ingroup_Loyalty", "Authority_Respect", "Purity_Sanctity"]
'''
MODELS = [
    "llama2_7b", "llama32_3b", "qwen25_7b", "mistral_7b",
    "llama31_8b", "qwen25_14b", "phi-3-mini",
    "mistral_7b_v3", "phi-3-small", "gpt-4o-mini", "gpt-4o", "gpt-52"
]
'''

MODELS = [
    "qwen25_7b", "qwen25_14b", "qwen25_32b", "qwen25_72b"
]
RESP_DIR = ".../politic_morality/data/llm_response"
RESP_TMPL = "morality_binary_conditioned_pct_with_responses_vote_{topic}_{model}.csv"
# # # morality_1samples_conditioned_pct_with_responses_{topic}_{model}.csv
# morality_binary_conditioned_pct_with_responses_{topic}_{model}.csv
# morality_binary_conditioned_pct_with_responses_third_person_{topic}_{model}.csv
# morality_binary_conditioned_pct_with_responses_vote_{topic}_{model}.csv
# # # morality_personas_conditioned_pct_with_responses_{topic}_{model}.csv

RESP_TMPL_BASELINE = "only_pct_with_responses_{model}.csv"

def topic_key(topic: str) -> str:
    return "_".join([topic.replace("/", "OR")])

def read_model_topic_csv(topic: str, model: str) -> pd.DataFrame | None:
    key = topic_key(topic)
    fpath = os.path.join(RESP_DIR, RESP_TMPL.format(n=N_PER_SCORE, topic=key, model=model))
    if not os.path.exists(fpath):
        fpath = os.path.join(RESP_DIR, RESP_TMPL.replace("1samples", "MFD_1samples").replace("binary", "MFD_binary").replace("personas", "MFD_personas").format(n=N_PER_SCORE, topic=key, model=model))
        if not os.path.exists(fpath):
            print(f"[WARN] skip {fpath}")
            return None
    try:
        return pd.read_csv(fpath)
    except Exception as e:
        print(f"[ERROR] {fpath}: {e}")
        return None

def read_model_baseline_csv(model: str) -> pd.DataFrame | None:
    fpath = os.path.join(RESP_DIR, RESP_TMPL_BASELINE.format(model=model))
    if not os.path.exists(fpath):
        print(f"[WARN] skip {fpath}")
        return None
    try:
        return pd.read_csv(fpath)
    except Exception as e:
        print(f"[ERROR] {fpath}: {e}")
        return None

def clean_pct_opinion(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    vals = []
    for s in tqdm(out["response_opinion"], total=len(out), desc="validate", leave=False):
        if validate_completion(s) == "valid":
            digit = next((ch for ch in str(s) if ch in "1234"), None)
            vals.append(digit)
        else:
            vals.append(None)
    out["pct_opinion"] = vals
    return out

def summarize_to_results(df_out: pd.DataFrame):
    rows = []
    for score, g in df_out.groupby("score"):
        g_sorted = g.sort_values("pct_id").reset_index(drop=True)
        choice_labels = g_sorted["pct_opinion"].tolist()
        econ, soc = calculate_pct_coordinates(choice_labels)
        n_items = sum(x not in [None, "unknown", "invalid"] for x in choice_labels)
        rows.append(dict(score=score, econ=econ, soc=soc, n_items=n_items))
    return pd.DataFrame(rows)


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm



import matplotlib.cm as cm
import matplotlib.patches as patches

def plot_extreme_arrows_grid(df_all: pd.DataFrame, topic: str, results_baseline: dict):

    df_all = df_all.copy()
    df_all["score"] = pd.to_numeric(df_all["score"], errors="coerce")
    df_all = df_all[df_all["score"].isin([-3, 3])]

    arrow_models = set(df_all["model"].unique())
    baseline_models = set(results_baseline.keys()) if results_baseline else set()
    model_list = sorted(arrow_models | baseline_models)
    if not model_list:
        print(f"[INFO] nothing to plot for {topic}")
        return

    cmap = cm.get_cmap("tab10", len(model_list))
    color_map = {m: cmap(i) for i, m in enumerate(model_list)}

    LIM = 10
    fig, ax = plt.subplots(figsize=(6.5, 6.5))

    
    quad_colors = {
        "UL": "#c7d8f0",  
        "UR": "#cbe7e2",  
        "LL": "#f3e3c7",  
        "LR": "#f6cfd1",  
    }

    
    ax.add_patch(patches.Rectangle(
        (-LIM, -LIM), LIM, LIM,
        facecolor=quad_colors["LL"], edgecolor="none", zorder=0, alpha=0.7
    ))
    
    ax.add_patch(patches.Rectangle(
        (0, -LIM), LIM, LIM,
        facecolor=quad_colors["LR"], edgecolor="none", zorder=0, alpha=0.7
    ))
    
    ax.add_patch(patches.Rectangle(
        (-LIM, 0), LIM, LIM,
        facecolor=quad_colors["UL"], edgecolor="none", zorder=0, alpha=0.7
    ))
    
    ax.add_patch(patches.Rectangle(
        (0, 0), LIM, LIM,
        facecolor=quad_colors["UR"], edgecolor="none", zorder=0, alpha=0.7
    ))

    ax.set_xlim(-LIM, LIM)
    ax.set_ylim(-LIM, LIM)
    ax.set_xticks(range(-LIM, LIM + 1, 2))
    ax.set_yticks(range(-LIM, LIM + 1, 2))
    ax.set_xticks(range(-LIM, LIM + 1, 1), minor=True)
    ax.set_yticks(range(-LIM, LIM + 1, 1), minor=True)

    ax.grid(which="major", color="#b0b0b0", alpha=0.35,
            linewidth=0.8, zorder=0)
    ax.grid(which="minor", color="#d0d0d0", alpha=0.25,
            linewidth=0.5, zorder=0)

    axis_color = "#444444"
    
    ax.annotate(
        "", xy=(0, LIM), xytext=(0, -LIM),
        arrowprops=dict(arrowstyle="<->", color=axis_color, lw=1.4),
        zorder=3
    )
    
    ax.annotate(
        "", xy=(LIM, 0), xytext=(-LIM, 0),
        arrowprops=dict(arrowstyle="<->", color=axis_color, lw=1.4),
        zorder=3
    )

    for spine in ax.spines.values():
        spine.set_visible(False)

    dx_list, dy_list = [], []
    for model, g in df_all.groupby("model"):
        g = g.sort_values("score")
        g_minus3 = g[g["score"] == -3]
        g_plus3  = g[g["score"] == 3]
        if g_minus3.empty or g_plus3.empty:
            continue

        x0, y0 = float(g_minus3["econ"].iloc[0]), float(g_minus3["soc"].iloc[0])
        x1, y1 = float(g_plus3["econ"].iloc[0]),  float(g_plus3["soc"].iloc[0])
        color = color_map[model]

        dx_list.append(x1 - x0)
        dy_list.append(y1 - y0)

        ax.scatter([x0, x1], [y0, y1],
                   color=color, s=32, zorder=4,
                   edgecolors="white", linewidths=0.6)

        if not (np.isclose(x0, x1) and np.isclose(y0, y1)):
            ax.annotate(
                "", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(
                    arrowstyle="-|>",
                    lw=1.5, color=color,
                    ls="-",
                    shrinkA=0, shrinkB=0,
                    mutation_scale=11
                ),
                zorder=3
            )

    if results_baseline:
        for model in model_list:
            d = results_baseline.get(model)
            if not d:
                continue
            bx, by = d.get("base_econ"), d.get("base_soc")
            if bx is None or by is None:
                continue
            ax.scatter(
                [bx], [by],
                marker="X", s=40,
                linewidths=0.9,
                edgecolors="black",
                facecolors="none",
                zorder=5
            )

    topic_name = topic.replace("_", " ")
    ax.text(LIM * 0.95, LIM * 0.95, topic_name,
            ha="right", va="top", fontsize=14, fontweight="bold")

    ax.text(0, LIM + 0.5, "Authoritarian",
            ha="center", va="bottom", fontsize=11)
    ax.text(0, -LIM - 1, "Libertarian",
            ha="center", va="top", fontsize=11)
    ax.text(-LIM - 1.1, 0, "Economic-Left",
            ha="right", va="center", fontsize=11, rotation=90)
    ax.text(LIM + 0.5, 0, "Economic-Right",
            ha="left", va="center", fontsize=11, rotation=90)

    ax.set_xlabel("")
    ax.set_ylabel("")

    arrow_models_sorted = sorted(arrow_models)

    legend_model_handles = [
        plt.Line2D(
            [0], [0],
            color=color_map[m],
            lw=1.5, ls="-", marker="o",
            markersize=5, label=m
        )
        for m in arrow_models_sorted
    ]
    baseline_handle = plt.Line2D(
        [0], [0],
        marker="X", lw=0, markersize=6,
        markeredgewidth=0.9,
        markeredgecolor="black",
        markerfacecolor="none",
        label="Baseline (PCT only)"
    )
    handles = legend_model_handles + [baseline_handle]

    ax.legend(
        handles=handles, title="Models",
        loc="upper left", fontsize=8, title_fontsize=9,
        frameon=True, facecolor="white",
        edgecolor="gray", fancybox=True
    )

    tmp = df_all[~df_all["model"].isin(["HUMAN","HUMAN_LIKERT"])]

    minus = tmp[tmp["score"] == -3].groupby("model")[["econ","soc"]].mean()
    plus  = tmp[tmp["score"] ==  3].groupby("model")[["econ","soc"]].mean()
    
    delta = (plus - minus).dropna()
    dx, dy = delta["econ"].mean(), delta["soc"].mean()

    
    ax.text(LIM * 0.25, -LIM * 0.25,
            fr"$\bar{{\Delta}} \approx ({dx:.1f}, {dy:.1f})$",
            fontsize=16, ha="left", va="center")

    if dx_list and dy_list:
        n_right = sum(d > 0 for d in dx_list)
        n_left  = sum(d < 0 for d in dx_list)
        n_up    = sum(d > 0 for d in dy_list)
        n_down  = sum(d < 0 for d in dy_list)

        ax_in = ax.inset_axes([0.80, 0.05, 0.10, 0.10])
        ax_in.set_xlim(-1, 1)
        ax_in.set_ylim(-1, 1)
        ax_in.set_xticks([])
        ax_in.set_yticks([])
        ax_in.set_aspect("equal", "box")

        ax_in.set_facecolor("white")
        for spine in ax_in.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor("gray")
            spine.set_linewidth(0.8)

        arrow_color = "#555555"

        ax_in.text(
                    0.5, 0.02, "Majority",        
                    transform=ax_in.transAxes,   
                    ha="center", va="bottom",    
                    fontsize=8
                )

        ax_in.plot([-0.5, 0.5], [0, 0], color=arrow_color, lw=0.8)
        ax_in.plot([0, 0], [-0.5, 0.5], color=arrow_color, lw=0.8)

        if n_right > n_left:
            ax_in.annotate(
                "", xy=(0.7, -0.045), xytext=(0.0, 0.0),  
                arrowprops=dict(arrowstyle="-|>", color=arrow_color, lw=0.7)
            )
        elif n_left > n_right:
            ax_in.annotate(
                "", xy=(-0.7, -0.045), xytext=(0.0, 0.0), 
                arrowprops=dict(arrowstyle="-|>", color=arrow_color, lw=0.7)
            )
    
        if n_up > n_down:
            ax_in.annotate(
                "", xy=(0.0, 0.7), xytext=(0.0, 0.0),   
                arrowprops=dict(arrowstyle="-|>", color=arrow_color, lw=0.7)
            )
        elif n_down > n_up:
            ax_in.annotate(
                "", xy=(0.0, -0.7), xytext=(0.0, 0.0), 
                arrowprops=dict(arrowstyle="-|>", color=arrow_color, lw=0.7)
            )





    plt.tight_layout()
    plt.savefig(f"./{topic}_MFQ_morally_personas_option_pct.pdf", dpi=300, bbox_inches="tight")
    plt.show()



import numpy as np
'''
MODEL_ORDER = [
    "llama2_7b",
    "llama32_3b",
    "llama31_8b",
    "mistral_7b",
    "mistral_7b_v3",
    "phi-3-mini",
    "phi-3-small",
    "qwen25_7b",
    "qwen25_14b",
    "gpt-4o-mini",
    "gpt-4o",
    "gpt-52"
]
'''

MODEL_ORDER = {
    "qwen25_7b",
    "qwen25_14b",
    "qwen25_32b",
    "qwen25_72b",
}

def compute_metrics_for_topic(topic: str):

    dx_list, dy_list = [], []

    econ_start_list = []  
    econ_end_list   = []  
    soc_start_list  = []
    soc_end_list    = []

    for model in MODEL_ORDER:
        df = read_model_topic_csv(topic, model)
        if df is None or df.empty:
            print(f"[WARN] no data for topic={topic}, model={model}")
            continue

        keep = ["topic", "score", "pct_id", "pct_question", "response_opinion"]
        if not all(c in df.columns for c in keep):
            print(f"[WARN] missing columns for topic={topic}, model={model}")
            continue

        df_out = clean_pct_opinion(df[keep])
        df_out["score"] = pd.to_numeric(df_out["score"], errors="coerce")

        df_res = summarize_to_results(df_out)
        g_minus = df_res[df_res["score"] == -3]
        g_plus  = df_res[df_res["score"] ==  3]
        if g_minus.empty or g_plus.empty:
            print(f"[WARN] no -3 or 3 for topic={topic}, model={model}")
            continue

        econ_start = float(g_minus["econ"].iloc[0])
        soc_start  = float(g_minus["soc"].iloc[0])
        econ_end   = float(g_plus["econ"].iloc[0])
        soc_end    = float(g_plus["soc"].iloc[0])

        econ_start_list.append(econ_start)
        econ_end_list.append(econ_end)
        soc_start_list.append(soc_start)
        soc_end_list.append(soc_end)

        dx_list.append(econ_end - econ_start)
        dy_list.append(soc_end  - soc_start)

    if not dx_list:
        print(f"[INFO] no usable Δ for topic={topic}")
        return

    start_e = np.array(econ_start_list)
    end_e   = np.array(econ_end_list)
    start_s = np.array(soc_start_list)
    end_s   = np.array(soc_end_list)
    
    dx = np.array(dx_list)
    dy = np.array(dy_list)

    mean_dx = dx.mean()
    mean_dy = dy.mean()

    def maj_from_array(arr):
        n_pos = np.sum(arr > 0)
        n_neg = np.sum(arr < 0)
        if n_pos > n_neg:
            return "+1"
        elif n_neg > n_pos:
            return "-1"
        else:
            return "0"

    D_e_mj = maj_from_array(dx)
    D_s_mj = maj_from_array(dy)



    # DirAlign
    mean_vec = np.array([dx.mean(), dy.mean()])
    norm_mean = np.linalg.norm(mean_vec)
    
    cos_list = []
    for dx_i, dy_i in zip(dx, dy):
        v_i = np.array([dx_i, dy_i])
        norm_i = np.linalg.norm(v_i)
        if norm_i == 0 or norm_mean == 0:
            continue  
        cos_i = float(np.dot(v_i, mean_vec) / (norm_i * norm_mean))
        cos_list.append(cos_i)
    
    if cos_list:
        dir_align = float(np.mean(cos_list))
    else:
        dir_align = float("nan")

    # “per-axis directional bias” 或 “signed directionality” in [-1,1]
    rho_e = float(np.mean(np.sign(dx)))      
    rho_s = float(np.mean(np.sign(dy)))

    
    cross_e = ((start_e < 0) & (end_e > 0)) | ((start_e > 0) & (end_e < 0))
    p_e = float(np.mean(cross_e))
    
    cross_s = ((start_s < 0) & (end_s > 0)) | ((start_s > 0) & (end_s < 0))
    p_s = float(np.mean(cross_s))

    
    # mean shift magnitude
    r_bar = float(np.mean(np.sqrt(dx**2 + dy**2)))


    # (1) Mean Resultant Length (MRL) of shift directions
    #     rho_dir = || mean_i ( Δ_i / ||Δ_i|| ) ||
    unit_vecs = []
    for dx_i, dy_i in zip(dx, dy):
        v_i = np.array([dx_i, dy_i], dtype=float)
        norm_i = float(np.linalg.norm(v_i))
        if norm_i == 0:
            continue
        unit_vecs.append(v_i / norm_i)

    if unit_vecs:
        unit_vecs = np.stack(unit_vecs, axis=0)              # [M, 2]
        rho_dir = float(np.linalg.norm(unit_vecs.mean(axis=0)))
    else:
        rho_dir = float("nan")

    # (2)(3) rejection / endorsement centroids
    v_rej = np.stack([start_e, start_s], axis=1)             # [N, 2]
    v_eds = np.stack([end_e, end_s], axis=1)                 # [N, 2]
    mu_rej = v_rej.mean(axis=0)                               # [2]
    mu_eds = v_eds.mean(axis=0)                               # [2]
    mu_rej_e, mu_rej_s = float(mu_rej[0]), float(mu_rej[1])
    mu_eds_e, mu_eds_s = float(mu_eds[0]), float(mu_eds[1])

    # (4)(5) positional dispersion (RMS distance to centroid)
    R_rej = float(np.sqrt(np.mean(np.sum((v_rej - mu_rej) ** 2, axis=1))))
    R_eds = float(np.sqrt(np.mean(np.sum((v_eds - mu_eds) ** 2, axis=1))))


    
    
    print("%", topic, "— majority + mean Δ")
    print(f"{D_e_mj} & {D_s_mj} & {rho_e:.2f} & {rho_s:.2f} & {mean_dx:.2f} & {mean_dy:.2f} \\\\")
    print()
    

    print("%", topic, "— rho_dir + frac + mean |Δ|")
    print(f"{p_e:.2f} & {p_s:.2f}  & {rho_dir:.2f} & {r_bar:.2f} \\\\")
    print()

    print("%", topic, "— centroids (rej / eds) + positional dispersion")
    print(
        f"{mu_rej_e:.2f} & {mu_rej_s:.2f} & {mu_eds_e:.2f} & {mu_eds_s:.2f} & {R_rej:.2f} & {R_eds:.2f} \\\\"
    )
    print()


def main():

    results_baseline = {} # [the result of non-moral-score conditioned]
    for model in MODELS:
        df_baseline = read_model_baseline_csv(model)
        df_baseline_out = clean_pct_opinion(df_baseline[["pct_id", "pct_question", "response_opinion"]])
        base_sorted = df_baseline_out.sort_values("pct_id").reset_index(drop=True)
        base_choice_labels = base_sorted["pct_opinion"].tolist()
        base_econ, base_soc = calculate_pct_coordinates(base_choice_labels)
        results_baseline[model] = {"base_econ": base_econ, "base_soc": base_soc}
    print(results_baseline)
    
    for topic in TOPICS:
        results_all = [] # [the result of each possible moral score]
        
        for model in MODELS:
            df = read_model_topic_csv(topic, model)
            if df is None or df.empty:
                continue

            keep = ["topic", "score", "pct_id", "pct_question", "response_opinion"]
            if not all(c in df.columns for c in keep):
                continue

            df_out = clean_pct_opinion(df[keep])
            df_out["model"] = model
            df_out["topic"] = topic

            df_res = summarize_to_results(df_out)
            if df_res.empty:
                continue
            df_res["model"] = model
            df_res["topic"] = topic
            
            results_all.append(df_res)

        if not results_all:
            print(f"[INFO] no data for {topic}")
            continue

        df_all = pd.concat(results_all, ignore_index=True)
        print(df_all)
        df_all = df_all[df_all["score"].isin([-3, 3])] 
        print(df_all)
        if df_all.empty:
            print(f"[INFO] no extreme scores for {topic}")
            continue

        plot_extreme_arrows(df_all, topic, results_baseline)
        plot_extreme_arrows_grid(df_all, topic, results_baseline)


if __name__ == "__main__":
    main()

    for topic in TOPICS:
        print(f"=== {topic} ===")
        compute_metrics_for_topic(topic)